# Setup

In [33]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.metrics import mean_squared_error, r2_score, mean_absolute_error, mean_absolute_percentage_error
from sklearn.model_selection import train_test_split, KFold, cross_val_score
from sklearn.linear_model import LinearRegression

In [53]:
# Custom
from utils.base import load_dataset
from utils.eda import check_dataset_size_and_shape, check_columns_for_nulls
from utils.tests import test_train_test_split_ratio

## Load Dataset

In [4]:
DATASET_FILENAME = "risk_assessment_emails.csv"

In [5]:

df = load_dataset(DATASET_FILENAME)

## Shuffle Reproducabity

Without random_state, you'd get a different split each run, producing different train/test sets and therefore different model metrics each time. Passing RANDOM_STATE makes it reproducible. This applies to sklearn `train_test_split`.

`np.random.seed(RANDOM_STATE)` seeds NumPy's global RNG. It matters when using `np.random.*` functions directly (e.g., `np.random.shuffle`, `np.random.choice`).

In [34]:
RANDOM_STATE: int = 42  # global seed for full reproducibility
np.random.seed(RANDOM_STATE)

# Exploratory Data Analysis (EDA)

In [6]:
check_dataset_size_and_shape(df)

Number of rows: 20
Dataset shape: (20, 1)
Number of rows (alternative): 20


In [7]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 20 entries, 0 to 19
Data columns (total 1 columns):
 #   Column  Non-Null Count  Dtype
---  ------  --------------  -----
 0   Emails  20 non-null     str  
dtypes: str(1)
memory usage: 292.0 bytes


In [8]:
df.describe()

,Emails
count,20
unique,20
top,An internal memo revealed testing issues that ...
freq,1


In [9]:
df.head()

,Emails
0,An internal memo revealed testing issues that ...
1,"The agency reported that CS2 had over 1,000 re..."
2,software failures ended up costing an estimate...
3,"EDS, a Texas-based contractor, also announced ..."
4,"As result of software failures, thousands of t..."


## Missing Data


In [10]:
check_columns_for_nulls(df)

Emails    False
dtype: bool


# Data Mining & Feature Engineering

Text mining via topic generation and feature engineering to create new columns before training (Minewiskan, n.d.). 

From AI:

"While data mining focuses on discovering hidden patterns in existing data, feature engineering focuses on preparing and modifying that data so machine learning models can understand it better. Data mining finds the insights; feature engineering builds the inputs."

## Topic Extraction

### Via Regex

Via regex and predefined keywords and sub keywords to look for.

In [11]:
import re

topic_keywords = {
  "Software failures": [
    # "software", "bug", "failure", "crash", "error", "defect", "outage", "system failed",
    "testing", "design", "end of life (eol)", "end of vendor support (eovs)", "vulnerability", "hacking/malware"
  ],
  "Hardware failures": [
    # "disk", "drive", "hardware", "short-circuit", "fire", "datacenter", "power", "backup power",
    "natural/man-made disasters", "hard drive failures", "network failures", "loss of power due to insufficient backup"
  ],
  "Users & computers": [
    # "user", "users", "computer", "accounts", "inactive user", "manager", "admin"
    "user accounts with passwords never expire", "user accounts with passwords not required to expire", "inactive user accounts"
  ],
  "Permissions": [
    # "permission", "permissions", "administrative", "admin rights", "security groups",
    # "shared folders", "accessible by everyone",
    "user accounts with administrative permissions",
    "empty security groups",
    "administrative groups"
  ],
  "Data": [
    # "data", "file", "files", "dataset", "records", "information",
    "shared folders accessible by everyone",
    "file names containing sensitive data"
  ],
}

def extract_topics(text: str):
  text = str(text).lower()
  matched = []
  for topic, keywords in topic_keywords.items():
    if any(re.search(rf"\b{re.escape(k.lower())}\b", text) for k in keywords):
      matched.append(topic)
  return matched if matched else ["Other"]

df["Topics"] = df["Emails"].apply(extract_topics)

# Preview
df[["Emails", "Topics"]].head(10)

,Emails,Topics
0,An internal memo revealed testing issues that ...,[Software failures]
1,"The agency reported that CS2 had over 1,000 re...",[Other]
2,software failures ended up costing an estimate...,[Software failures]
3,"EDS, a Texas-based contractor, also announced ...",[Software failures]
4,"As result of software failures, thousands of t...",[Software failures]
5,Baggage handling system failed miserably in re...,[Software failures]
6,"At the end of a busy day, the manager receives...",[Other]
7,"For missed EOVS notice, Technology Manager rec...",[Other]
8,"Due to Insufficient backup power, there is a 7...",[Other]
9,Disk drive failure caused Operating System ero...,[Other]


In [12]:
df["Topics"].value_counts()

Topics
[Other]                9
[Software failures]    5
[Permissions]          3
[Data]                 2
[Users & computers]    1
Name: count, dtype: int64


### Via LDA

Uses LDA (Latent Dirichlet Allocation), an unsupervised learning algorithm for hidden textual topic.

In [16]:
import re
from sklearn.feature_extraction.text import CountVectorizer
from sklearn.decomposition import LatentDirichletAllocation

def clean_text(text):
    text = str(text).lower()
    text = re.sub(r'http\S+|www\S+', '', text)  # remove URLs
    text = re.sub(r'@\w+', '', text)             # remove @mentions
    text = re.sub(r'[^a-z\s]', '', text)         # letters only
    return text.strip()

def extract_topics(df, text_column='tweet_text', n_topics=5):
  df['clean_text_lda'] = df[text_column].apply(clean_text)

  count_vec = CountVectorizer(stop_words='english', max_features=500, min_df=2)
  dtm = count_vec.fit_transform(df['clean_text_lda'])

  lda = LatentDirichletAllocation(n_components=n_topics, random_state=42)
  lda.fit(dtm)

  feature_names = count_vec.get_feature_names_out()
  for idx, topic in enumerate(lda.components_):
      top_words = [feature_names[i] for i in topic.argsort()[:-9:-1]]
      print(f"Topic {idx+1}: {', '.join(top_words)}")

  df['topic_lda'] = lda.transform(dtm).argmax(axis=1)
  # return df[[text_column, 'topic_lda']].head(10)
  return df

df_lda = extract_topics(df, text_column='Emails', n_topics=5)
df_lda[["Emails", "topic_lda"]].head(10)

Topic 1: user, accounts, administrative, groups, category, passwords, security, admin
Topic 2: hardware, failures, caused, disk, operating, resulted, failed, failure
Topic 3: software, admin, failure, failed, caused, thousands, resulted, testing
Topic 4: software, testing, bad, million, failures, failure, thousands, caused
Topic 5: software, failure, manager, email, receives, week, chance, failures


,Emails,topic_lda
0,An internal memo revealed testing issues that ...,3
1,"The agency reported that CS2 had over 1,000 re...",4
2,software failures ended up costing an estimate...,3
3,"EDS, a Texas-based contractor, also announced ...",3
4,"As result of software failures, thousands of t...",3
5,Baggage handling system failed miserably in re...,2
6,"At the end of a busy day, the manager receives...",4
7,"For missed EOVS notice, Technology Manager rec...",4
8,"Due to Insufficient backup power, there is a 7...",1
9,Disk drive failure caused Operating System ero...,1


## Subtopics

In [13]:
# Yes — add a new column with matched sub-topics (keywords)
def extract_subtopics(text: str):
  text = str(text).lower()
  matched = []

  for _, keywords in topic_keywords.items():
    for kw in keywords:
      if re.search(rf"\b{re.escape(kw.lower())}\b", text):
        matched.append(kw)

  # remove duplicates while preserving order
  matched = list(dict.fromkeys(matched))
  return matched if matched else ["Other"]

df["Subtopics"] = df["Emails"].apply(extract_subtopics)

# Preview
df[["Emails", "Topics", "Subtopics"]].head(10)

,Emails,Topics,Subtopics
0,An internal memo revealed testing issues that ...,[Software failures],[testing]
1,"The agency reported that CS2 had over 1,000 re...",[Other],[Other]
2,software failures ended up costing an estimate...,[Software failures],"[testing, design]"
3,"EDS, a Texas-based contractor, also announced ...",[Software failures],[testing]
4,"As result of software failures, thousands of t...",[Software failures],[testing]
5,Baggage handling system failed miserably in re...,[Software failures],[testing]
6,"At the end of a busy day, the manager receives...",[Other],[Other]
7,"For missed EOVS notice, Technology Manager rec...",[Other],[Other]
8,"Due to Insufficient backup power, there is a 7...",[Other],[Other]
9,Disk drive failure caused Operating System ero...,[Other],[Other]


In [14]:
df["Subtopics"].explode()

0                                           testing
1                                             Other
2                                           testing
2                                            design
3                                           testing
4                                           testing
5                                           testing
6                                             Other
7                                             Other
8                                             Other
9                                             Other
10                                            Other
11                                            Other
12                                            Other
13                                            Other
14                           inactive user accounts
15    user accounts with administrative permissions
16                            empty security groups
17                            administrative groups
18          

In [15]:
df["Subtopics"].explode().value_counts()

Subtopics
Other                                            9
testing                                          5
design                                           1
inactive user accounts                           1
user accounts with administrative permissions    1
empty security groups                            1
administrative groups                            1
shared folders accessible by everyone            1
file names containing sensitive data             1
Name: count, dtype: int64

# Impact, Likelyhood & Risk Factor

In [17]:
# Calculate likelihood (frequency of each topic)
topic_likelihood = df["Topics"].explode().value_counts(normalize=True).to_dict()

# Define impact scores for each topic (customize based on your risk assessment)
impact_scores = {
  "Software failures": 8,
  "Hardware failures": 7,
  "Users & computers": 5,
  "Permissions": 9,
  "Data": 9,
  "Other": 3
}

# Calculate impact for each row
df["Impact"] = df["Topics"].apply(lambda topics: max([impact_scores.get(t, 0) for t in topics]))

# Calculate likelihood for each row (as a percentage)
df["Likelihood"] = df["Topics"].apply(lambda topics: max([topic_likelihood.get(t, 0) for t in topics]))

# Calculate risk factor (Likelihood × Impact)
df["Risk_factor"] = df["Likelihood"] * df["Impact"]

print(df[["Topics", "Likelihood", "Impact", "Risk_factor"]].head(10))

                Topics  Likelihood  Impact  Risk_factor
0  [Software failures]        0.25       8         2.00
1              [Other]        0.45       3         1.35
2  [Software failures]        0.25       8         2.00
3  [Software failures]        0.25       8         2.00
4  [Software failures]        0.25       8         2.00
5  [Software failures]        0.25       8         2.00
6              [Other]        0.45       3         1.35
7              [Other]        0.45       3         1.35
8              [Other]        0.45       3         1.35
9              [Other]        0.45       3         1.35


In [18]:
df["Impact"].value_counts()

Impact
3    9
8    5
9    5
5    1
Name: count, dtype: int64

In [19]:
df["Likelihood"].value_counts()

Likelihood
0.45    9
0.25    5
0.15    3
0.10    2
0.05    1
Name: count, dtype: int64

In [20]:
df["Risk_factor"].value_counts()

Risk_factor
1.35    9
2.00    5
1.35    3
0.90    2
0.25    1
Name: count, dtype: int64

# Probability, Probability_Label & Risk Control Present

In [22]:
# Probability: extract from email text if present, otherwise infer from events, otherwise default to Medium
probability_word_map = {
  "low": 0.30,
  "medium": 0.50,
  "high": 0.70,
}

def get_probability(row):
  text = str(row["Emails"]).lower()

  # 1) Extract explicit percentage (e.g., "70%")
  pct_match = re.search(r'(\d+(?:\.\d+)?)\s*%', text)
  if pct_match:
    return max(0.0, min(1.0, float(pct_match.group(1)) / 100.0))

  # 2) Extract qualitative probability words
  for word, val in probability_word_map.items():
    if re.search(rf"\b{word}\b", text):
      return val

  # 3) Infer from mapped topic likelihood
  topics = row.get("Topics", [])
  if isinstance(topics, list) and topics:
    inferred = [topic_likelihood.get(t) for t in topics if t in topic_likelihood]
    if inferred:
      return max(inferred)

  # 4) Default to Medium
  return 0.50

def probability_label(p):
  if p >= 0.67:
    return "High"
  if p >= 0.34:
    return "Medium"
  return "Low"

# Risk control presence: simple keyword-based detection
control_keywords = [
  "control", "mitigation", "backup", "redundancy", "monitoring", "alert",
  "patch", "firewall", "mfa", "encryption", "policy", "procedure",
  "training", "audit", "incident response", "access control"
]
# e.g. '\\b(control|mitigation|backup|r...'
control_pattern = r"\b(" + "|".join(re.escape(k) for k in control_keywords) + r")\b"

df["Probability"] = df.apply(get_probability, axis=1)
df["Probability_Label"] = df["Probability"].apply(probability_label)
df["Risk Control Present"] = df["Emails"].astype(str).str.lower().str.contains(control_pattern, regex=True)

df[["Emails", "Topics", "Probability", "Probability_Label", "Risk Control Present"]].head(10)

/var/folders/yh/2c6g86wn0_1_v36xltfm8dgh0000gn/T/ipykernel_2740/3204604417.py:49: UserWarning: This pattern is interpreted as a regular expression, and has match groups. To actually get the groups, use str.extract.
  df["Risk Control Present"] = df["Emails"].astype(str).str.lower().str.contains(control_pattern, regex=True)


,Emails,Topics,Probability,Probability_Label,Risk Control Present
0,An internal memo revealed testing issues that ...,[Software failures],0.25,Low,False
1,"The agency reported that CS2 had over 1,000 re...",[Other],0.70,High,False
2,software failures ended up costing an estimate...,[Software failures],0.25,Low,False
3,"EDS, a Texas-based contractor, also announced ...",[Software failures],0.25,Low,False
4,"As result of software failures, thousands of t...",[Software failures],0.25,Low,False
5,Baggage handling system failed miserably in re...,[Software failures],0.25,Low,False
6,"At the end of a busy day, the manager receives...",[Other],0.45,Medium,False
7,"For missed EOVS notice, Technology Manager rec...",[Other],0.45,Medium,False
8,"Due to Insufficient backup power, there is a 7...",[Other],0.70,High,True
9,Disk drive failure caused Operating System ero...,[Other],0.45,Medium,False


In [24]:
control_pattern

'\\b(control|mitigation|backup|redundancy|monitoring|alert|patch|firewall|mfa|encryption|policy|procedure|training|audit|incident\\ response|access\\ control)\\b'

# Preprocessing

In [25]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 20 entries, 0 to 19
Data columns (total 11 columns):
 #   Column                Non-Null Count  Dtype  
---  ------                --------------  -----  
 0   Emails                20 non-null     str    
 1   Topics                20 non-null     object 
 2   Subtopics             20 non-null     object 
 3   clean_text_lda        20 non-null     str    
 4   topic_lda             20 non-null     int64  
 5   Impact                20 non-null     int64  
 6   Likelihood            20 non-null     float64
 7   Risk_factor           20 non-null     float64
 8   Probability           20 non-null     float64
 9   Probability_Label     20 non-null     str    
 10  Risk Control Present  20 non-null     bool   
dtypes: bool(1), float64(3), int64(2), object(2), str(3)
memory usage: 1.7+ KB


## Renaming

In [26]:
rename_map = {"Topics": "events", "Subtopics": "subevents"}

df.rename(columns=rename_map, inplace=True)
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 20 entries, 0 to 19
Data columns (total 11 columns):
 #   Column                Non-Null Count  Dtype  
---  ------                --------------  -----  
 0   Emails                20 non-null     str    
 1   events                20 non-null     object 
 2   subevents             20 non-null     object 
 3   clean_text_lda        20 non-null     str    
 4   topic_lda             20 non-null     int64  
 5   Impact                20 non-null     int64  
 6   Likelihood            20 non-null     float64
 7   Risk_factor           20 non-null     float64
 8   Probability           20 non-null     float64
 9   Probability_Label     20 non-null     str    
 10  Risk Control Present  20 non-null     bool   
dtypes: bool(1), float64(3), int64(2), object(2), str(3)
memory usage: 1.7+ KB


In [27]:
df[["events", "subevents"]].head()

,events,subevents
0,[Software failures],[testing]
1,[Other],[Other]
2,[Software failures],"[testing, design]"
3,[Software failures],[testing]
4,[Software failures],[testing]


## Encoding Categaricals

In [ ]:
from sklearn.preprocessing import OrdinalEncoder

string_cols = df.select_dtypes(include=["object", "string"]).columns.tolist()

# this catches other types of categorical data (lists, dicts) and normalizes them to strings before encoding
# Note: there might be a better ways to handle this.
def normalize_categorical_value(v):
    if isinstance(v, list):
        return " | ".join(map(str, v))
    if isinstance(v, dict):
        return " | ".join(f"{k}:{v[k]}" for k in sorted(v))
    return "" if v is None else str(v)

for col in string_cols:
    df[col] = df[col].apply(normalize_categorical_value)

if string_cols:
    enc = OrdinalEncoder(handle_unknown="use_encoded_value", unknown_value=-1)
    df[string_cols] = enc.fit_transform(df[string_cols])

In [41]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 20 entries, 0 to 19
Data columns (total 11 columns):
 #   Column                Non-Null Count  Dtype  
---  ------                --------------  -----  
 0   Emails                20 non-null     float64
 1   events                20 non-null     float64
 2   subevents             20 non-null     float64
 3   clean_text_lda        20 non-null     float64
 4   topic_lda             20 non-null     int64  
 5   Impact                20 non-null     int64  
 6   Likelihood            20 non-null     float64
 7   Risk_factor           20 non-null     float64
 8   Probability           20 non-null     float64
 9   Probability_Label     20 non-null     float64
 10  Risk Control Present  20 non-null     bool   
dtypes: bool(1), float64(8), int64(2)
memory usage: 1.7 KB


In [42]:
df.head()

,Emails,events,subevents,clean_text_lda,topic_lda,Impact,Likelihood,Risk_factor,Probability,Probability_Label,Risk Control Present
0,3.0,3.0,6.0,3.0,3,8,0.25,2.00,0.25,1.0,False
1,16.0,1.0,0.0,17.0,4,3,0.45,1.35,0.70,0.0,False
2,19.0,3.0,7.0,16.0,3,8,0.25,2.00,0.25,1.0,False
3,9.0,3.0,6.0,9.0,3,8,0.25,2.00,0.25,1.0,False
4,4.0,3.0,6.0,4.0,3,8,0.25,2.00,0.25,1.0,False


In [ ]:
# Gen via GPT-5.3-Codex.
# check for any remaining list/dict/tuple/set values in object/string columns that would break sklearn encoders
# Use this validation cell to confirm there are no list/dict values left in categorical columns before encoding

import pandas as pd

obj_cols = df.select_dtypes(include=["object", "string"]).columns.tolist()

def is_container_like(x):
    return isinstance(x, (list, dict, tuple, set))

issues = []
for c in obj_cols:
    mask = df[c].apply(is_container_like)
    n = int(mask.sum())
    if n > 0:
        sample = df.loc[mask, c].head(3).tolist()
        issues.append({"column": c, "rows_with_container_values": n, "sample_values": sample})

if issues:
    print("Columns that will break sklearn encoders:")
    print(pd.DataFrame(issues))
else:
    print("OK: No list/dict/tuple/set values found in object/string columns.")

OK: No list/dict/tuple/set values found in object/string columns.


# Exploratory Data Analysis (EDA)

In [44]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 20 entries, 0 to 19
Data columns (total 11 columns):
 #   Column                Non-Null Count  Dtype  
---  ------                --------------  -----  
 0   Emails                20 non-null     float64
 1   events                20 non-null     float64
 2   subevents             20 non-null     float64
 3   clean_text_lda        20 non-null     float64
 4   topic_lda             20 non-null     int64  
 5   Impact                20 non-null     int64  
 6   Likelihood            20 non-null     float64
 7   Risk_factor           20 non-null     float64
 8   Probability           20 non-null     float64
 9   Probability_Label     20 non-null     float64
 10  Risk Control Present  20 non-null     bool   
dtypes: bool(1), float64(8), int64(2)
memory usage: 1.7 KB


In [47]:
df.describe()

,Emails,events,subevents,clean_text_lda,topic_lda,Impact,Likelihood,Risk_factor,Probability,Probability_Label
count,20.00000,20.000000,20.000000,20.00000,20.000000,20.000000,20.000000,20.000000,20.000000,20.000000
mean,9.50000,1.700000,2.700000,9.50000,1.600000,5.850000,0.300000,1.412500,0.325000,1.250000
std,5.91608,1.128576,2.957595,5.91608,1.535544,2.777257,0.149561,0.438261,0.190221,0.638666
min,0.00000,0.000000,0.000000,0.00000,0.000000,3.000000,0.050000,0.250000,0.050000,0.000000
25%,4.75000,1.000000,0.000000,4.75000,0.000000,3.000000,0.150000,1.350000,0.150000,1.000000
50%,9.50000,1.000000,1.500000,9.50000,1.000000,6.500000,0.250000,1.350000,0.250000,1.000000
75%,14.25000,3.000000,6.000000,14.25000,3.000000,8.250000,0.450000,1.512500,0.450000,2.000000
max,19.00000,4.000000,8.000000,19.00000,4.000000,9.000000,0.450000,2.000000,0.700000,2.000000


In [48]:
df.head()

,Emails,events,subevents,clean_text_lda,topic_lda,Impact,Likelihood,Risk_factor,Probability,Probability_Label,Risk Control Present
0,3.0,3.0,6.0,3.0,3,8,0.25,2.00,0.25,1.0,False
1,16.0,1.0,0.0,17.0,4,3,0.45,1.35,0.70,0.0,False
2,19.0,3.0,7.0,16.0,3,8,0.25,2.00,0.25,1.0,False
3,9.0,3.0,6.0,9.0,3,8,0.25,2.00,0.25,1.0,False
4,4.0,3.0,6.0,4.0,3,8,0.25,2.00,0.25,1.0,False


# Normalization

# Train/Test Split

In [49]:
# get pd.Series/Vector for predictors
# ref @ https://pandas.pydata.org/docs/reference/api/pandas.DataFrame.drop.html
# Assumes numeric columns only after dropping non-numeric ones earlier in feature selection
X = df.drop(columns=['Risk_factor'])

# get pd.Series/Vector for target
y = df['Risk_factor']

# split 80/20
# per https://scikit-learn.org/stable/modules/generated/sklearn.model_selection.train_test_split.html
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=RANDOM_STATE)


In [55]:
test_train_test_split_ratio(df, X_train, X_test, expected_test_size=0.2)

Number of rows: 20
Dataset shape: (20, 11)
Number of rows (alternative): 20
Train size: 16, Test size: 4, Test ratio: 20.00%


In [50]:
X_train.info()

<class 'pandas.DataFrame'>
Index: 16 entries, 8 to 6
Data columns (total 10 columns):
 #   Column                Non-Null Count  Dtype  
---  ------                --------------  -----  
 0   Emails                16 non-null     float64
 1   events                16 non-null     float64
 2   subevents             16 non-null     float64
 3   clean_text_lda        16 non-null     float64
 4   topic_lda             16 non-null     int64  
 5   Impact                16 non-null     int64  
 6   Likelihood            16 non-null     float64
 7   Probability           16 non-null     float64
 8   Probability_Label     16 non-null     float64
 9   Risk Control Present  16 non-null     bool   
dtypes: bool(1), float64(7), int64(2)
memory usage: 1.3 KB


# Training

## Linear Regression

In [51]:
# https://scikit-learn.org/stable/auto_examples/linear_model/plot_ols_ridge.html
regressor = LinearRegression().fit(X_train, y_train)
# y_pred = regressor.predict(X_test)

## Support Vector Machine

Via a regressor model (SVR) since target is continous and not categorical.

In [58]:
from sklearn.pipeline import make_pipeline
from sklearn.svm import SVR
from sklearn.preprocessing import StandardScaler

svr = make_pipeline(
    StandardScaler(),
    SVR(kernel="rbf", C=1.0, epsilon=0.1)
)

svr.fit(X_train, y_train)
# y_pred = svr.predict(X_test)

,"steps steps: list of tuplesList of (name of step, estimator) tuples that are to be chained insequential order. To be compatible with the scikit-learn API, all stepsmust define `fit`. All non-last steps must also define `transform`. See:ref:`Combining Estimators <combining_estimators>` for more details.","[('standardscaler', ...), ('svr', ...)]"
,"transform_input transform_input: list of str, default=NoneThe names of the :term:`metadata` parameters that should be transformed by thepipeline before passing it to the step consuming it.This enables transforming some input arguments to ``fit`` (other than ``X``)to be transformed by the steps of the pipeline up to the step which requiresthem. Requirement is defined via :ref:`metadata routing <metadata_routing>`.For instance, this can be used to pass a validation set through the pipeline.You can only set this if metadata routing is enabled, which youcan enable using ``sklearn.set_config(enable_metadata_routing=True)``... versionadded:: 1.6",None
,"memory memory: str or object with the joblib.Memory interface, default=NoneUsed to cache the fitted transformers of the pipeline. The last stepwill never be cached, even if it is a transformer. By default, nocaching is performed. If a string is given, it is the path to thecaching directory. Enabling caching triggers a clone of the transformersbefore fitting. Therefore, the transformer instance given to thepipeline cannot be inspected directly. Use the attribute ``named_steps``or ``steps`` to inspect estimators within the pipeline. Caching thetransformers is advantageous when fitting is time consuming. See:ref:`sphx_glr_auto_examples_neighbors_plot_caching_nearest_neighbors.py`for an example on how to enable caching.",None
,"verbose verbose: bool, default=FalseIf True, the time elapsed while fitting each step will be printed as itis completed.",False
Name,Type,Value
"feature_names_in_ feature_names_in_: ndarray of shape (`n_features_in_`,)Names of features seen during :term:`fit`. Only defined if theunderlying estimator exposes such an attribute when fit... versionadded:: 1.0","ndarray[object](10,)","['Emails','events','subevents',...,'Probability','Probability_Label', 'Risk Control Present']"
n_features_in_ n_features_in_: intNumber of features seen during :term:`fit`. Only defined if theunderlying first estimator in `steps` exposes such an attributewhen fit... versionadded:: 0.24,int,10
,"copy copy: bool, default=TrueIf False, try to avoid a copy and do inplace scaling instead.This is not guaranteed to always work inplace; e.g. if the data isnot a NumPy array or scipy.sparse CSR matrix, a copy may still bereturned.",True
,"with_mean with_mean: bool, default=TrueIf True, center the data before scaling.This does not work (and will raise an exception) when attempted onsparse matrices, because centering them entails building a densematrix which in common use cases is likely to be too large to fit inmemory.",True
,"with_std with_std: bool, default=TrueIf True, scale the data to unit variance (or equivalently,unit standard deviation).",True
Name,Type,Value


# Evaluation

## For Linear Regression 

In [56]:

def evaluate_model(model, X_test, y_test):
  y_pred = model.predict(X_test)
  
  mse = mean_squared_error(y_test, y_pred)
  rmse = np.sqrt(mse)
  mae = mean_absolute_error(y_test, y_pred)
  r2 = r2_score(y_test, y_pred)
  mape = mean_absolute_percentage_error(y_test, y_pred)

  test_mse = float(round(mse,3))
  test_rmse = float(round(rmse,3))
  test_mae = float(round(mae,3))
  test_r2 = float(round(r2,3))
  test_mape = float(round(mape * 100, 3))  # convert to percentage
  
  # return test_mse, test_mae, test_r2
  return {
    'mse': test_mse,
    'rmse': test_rmse,
    'mae': test_mae,
    'r2': test_r2,
    'mape': test_mape,
  }

metrics = evaluate_model(regressor, X_test, y_test)
print(f"Test MSE: {metrics['mse']}")
print(f"Test RMSE: {metrics['rmse']}")
print(f"Test MAE: {metrics['mae']}")
print(f"Test R²: {metrics['r2']}")
print(f"Test MAPE: {metrics['mape']}%")

Test MSE: 3.301
Test RMSE: 1.817
Test MAE: 0.908
Test R²: -40.668
Test MAPE: 67.29%


In [59]:
metrics_svr = evaluate_model(svr, X_test, y_test)
print(f"Test MSE: {metrics_svr['mse']}")
print(f"Test RMSE: {metrics_svr['rmse']}")
print(f"Test MAE: {metrics_svr['mae']}")
print(f"Test R²: {metrics_svr['r2']}")
print(f"Test MAPE: {metrics_svr['mape']}%")

Test MSE: 0.008
Test RMSE: 0.088
Test MAE: 0.08
Test R²: 0.902
Test MAPE: 5.098%


# References

